#  F1 2023 Sessions — Complete Machine Learning Project

**Dataset:** `sessions.csv` — Formula 1 2023 Season Session Data  
**Author:** Mouhcine karimane 
**Course:** Machine Learning  

---

## Project Overview

This notebook is a **complete machine learning project** built step by step. The dataset contains information about every session (Practice, Qualifying, Race, Sprint) from the 2023 Formula 1 World Championship.

### What we will do:
1.  Load & explore the data
2.  Clean and preprocess
3.  Exploratory Data Analysis (EDA)
4.  Feature Engineering
5.  Clustering (K-Means, K-Means++, Hierarchical, DBSCAN)
6.  Classification (KNN, Decision Tree, Logistic Regression, Random Forest, AdaBoost, Stacking)
7.  Model Evaluation & Comparison
8.  Final Conclusion

---
## Section 0 — Install & Import Libraries

We import all the libraries we need at the top. This is good coding practice!

In [ ]:
#  Core libraries 
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#  Visualisation 
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

#  Preprocessing 
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score

#  Clustering 
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

#  Classification 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, StackingClassifier
)

#  Evaluation 
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

#  Style 
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

print('All libraries imported successfully!')

---
##  Section 1 — Load the Dataset

We start by loading the CSV file and getting a first look at what it contains.

> **Tip for students:** `.head()` shows the first 5 rows, `.info()` shows column types and null counts, and `.describe()` gives statistics for numeric columns.

In [ ]:
#  Load the dataset 
# If you are running this in Google Colab, upload the file first
# using the file panel on the left, then run this cell.
df = pd.read_csv('sessions.csv')

print(f' Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print()
df.head()

In [ ]:
#  Column types and non-null counts 
df.info()

In [ ]:
#  Statistical summary of numeric columns 
df.describe().round(2)

In [ ]:
#  Unique values in categorical columns 
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f'  {col:25s}: {df[col].nunique()} unique → {list(df[col].unique())[:6]}')

---
##  Section 2 — Data Cleaning

Before we do any analysis or modelling, we must clean the data:
- Check for **missing values** (NaN)
- Check for **duplicate rows**
- **Parse dates** properly
- **Encode categorical** columns for algorithms that need numbers

> **Why does it matter?** Messy data leads to wrong models. "Garbage in, garbage out!"

In [ ]:
#  Missing values 
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing cells: {missing.sum()}')

# Visualise missing values
fig, ax = plt.subplots(figsize=(10, 4))
missing.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Missing Values per Column', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#  Duplicates 
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')
if dupes > 0:
    df = df.drop_duplicates()
    print(f'Dropped duplicates. New shape: {df.shape}')
else:
    print(' No duplicates found!')

In [ ]:
#  Parse datetime columns 
df['date_start'] = pd.to_datetime(df['date_start'], utc=True)
df['date_end']   = pd.to_datetime(df['date_end'],   utc=True)

# Derive useful time features
df['duration_minutes'] = (df['date_end'] - df['date_start']).dt.total_seconds() / 60
df['month']            = df['date_start'].dt.month
df['day_of_week']      = df['date_start'].dt.dayofweek   # 0=Mon … 6=Sun
df['hour_start']       = df['date_start'].dt.hour

# GMT offset → numeric hours
def parse_gmt(s):
    sign = -1 if str(s).startswith('-') else 1
    parts = str(s).lstrip('-').split(':')
    return sign * (int(parts[0]) + int(parts[1]) / 60)

df['gmt_offset_hours'] = df['gmt_offset'].apply(parse_gmt)

print('Dates parsed. New features created:')
print(df[['duration_minutes', 'month', 'day_of_week',
          'hour_start', 'gmt_offset_hours']].head())

In [ ]:
#  Label Encoding for categorical columns 
# We create a clean copy with numeric codes (needed for ML models)
le = LabelEncoder()
encode_cols = ['session_type', 'session_name', 'country_code',
               'circuit_short_name', 'location']

df_encoded = df.copy()
label_maps = {}
for col in encode_cols:
    df_encoded[f'{col}_enc'] = le.fit_transform(df[col])
    label_maps[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print(' Encoding done. Example  session_type mapping:')
print(label_maps['session_type'])

---
##  Section 3 — Exploratory Data Analysis (EDA)

EDA means **exploring the data visually** before building models. It helps us:
- Understand distributions
- Spot patterns and relationships
- Find outliers

> **Think of EDA as getting to know your data before you start working with it.**

In [ ]:
#  3.1 Session type distribution 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
order = df['session_type'].value_counts().index
sns.countplot(data=df, x='session_type', ax=axes[0], order=order,
              palette='Set2', edgecolor='white')
axes[0].set_title('Sessions by Type', fontweight='bold')
axes[0].set_xlabel('Session Type')
axes[0].set_ylabel('Count')

# Pie chart
counts = df['session_type'].value_counts()
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('Set2', len(counts)))
axes[1].set_title('Session Type Proportion', fontweight='bold')

plt.suptitle('Session Type Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#  3.2 Histograms of numeric features 
num_features = ['duration_minutes', 'month', 'day_of_week',
                'hour_start', 'gmt_offset_hours',
                'circuit_key', 'country_key']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df_encoded[col].dropna(), bins=20,
                 color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Histograms of Numeric Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  3.3 Session duration by type (box plot) 
fig, ax = plt.subplots(figsize=(10, 5))
order = ['Practice', 'Qualifying', 'Race']
palette = {'Practice': '#4C72B0', 'Qualifying': '#DD8452', 'Race': '#55A868'}
sns.boxplot(data=df, x='session_type', y='duration_minutes',
            order=order, palette=palette, ax=ax, width=0.5)
sns.stripplot(data=df, x='session_type', y='duration_minutes',
              order=order, color='black', alpha=0.3, size=4, ax=ax)
ax.set_title('Session Duration by Type (minutes)', fontsize=13, fontweight='bold')
ax.set_xlabel('Session Type')
ax.set_ylabel('Duration (minutes)')
plt.tight_layout()
plt.show()

In [ ]:
#  3.4 Correlation Heatmap 
# Only numeric columns
numeric_df = df_encoded.select_dtypes(include=[np.number]).drop(
    columns=['meeting_key', 'session_key'], errors='ignore'
)

corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n Key insight: Values close to +1 or -1 show strong relationships between features.")

In [ ]:
#  3.5 Scatter Plots 
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Scatter 1: Duration vs Hour Start, coloured by session type
for stype, grp in df.groupby('session_type'):
    axes[0].scatter(grp['hour_start'], grp['duration_minutes'],
                    label=stype, alpha=0.7, s=60)
axes[0].set_xlabel('Session Start Hour (UTC)')
axes[0].set_ylabel('Duration (min)')
axes[0].set_title('Duration vs Start Hour', fontweight='bold')
axes[0].legend()

# Scatter 2: Month vs Duration
for stype, grp in df.groupby('session_type'):
    axes[1].scatter(grp['month'], grp['duration_minutes'],
                    label=stype, alpha=0.7, s=60)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Duration (min)')
axes[1].set_title('Duration vs Month', fontweight='bold')
axes[1].set_xticks(range(1, 13))
axes[1].legend()

# Scatter 3: GMT Offset vs Duration
for stype, grp in df.groupby('session_type'):
    axes[2].scatter(grp['gmt_offset_hours'], grp['duration_minutes'],
                    label=stype, alpha=0.7, s=60)
axes[2].set_xlabel('GMT Offset (hours)')
axes[2].set_ylabel('Duration (min)')
axes[2].set_title('Duration vs GMT Offset', fontweight='bold')
axes[2].legend()

plt.suptitle('Scatter Plot Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  3.6 Distribution Analysis (KDE plots) 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, color in zip(
    axes,
    ['duration_minutes', 'hour_start', 'gmt_offset_hours'],
    ['#4C72B0', '#DD8452', '#55A868']
):
    sns.histplot(df[col], kde=True, ax=ax, color=color,
                 bins=20, edgecolor='white')
    ax.set_title(f'Distribution of {col}', fontweight='bold')
    ax.set_xlabel(col.replace('_', ' ').title())
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--',
               label=f'Mean = {mean_val:.1f}')
    ax.legend()

plt.suptitle('Distribution Analysis with KDE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  3.7 Sessions per month 
monthly = df.groupby(['month', 'session_type']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
monthly.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.75)
ax.set_title('Number of Sessions per Month by Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Sessions')
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
ax.set_xticklabels([month_names[m-1] for m in monthly.index], rotation=0)
ax.legend(title='Session Type')
plt.tight_layout()
plt.show()

---
##  Section 4 — Feature Engineering

Feature Engineering means **creating new, more informative columns** from existing ones.  
Good features → better models.

> **Examples:** Instead of a raw timestamp, we extracted month, hour, and day-of-week. Now we add even more useful features.

In [ ]:
#  Feature: Is the session on a weekend? 
df_encoded['is_weekend'] = df_encoded['day_of_week'].apply(
    lambda x: 1 if x >= 5 else 0
)

#  Feature: Season part (early / mid / late) 
def season_part(month):
    if month <= 4:   return 0   # early season
    elif month <= 8: return 1   # mid season
    else:            return 2   # late season

df_encoded['season_part'] = df_encoded['month'].apply(season_part)

#  Feature: Session duration category 
# Short < 60 min | Medium 60–90 | Long > 90
def dur_category(d):
    if d < 60:  return 0
    elif d < 90: return 1
    else:        return 2

df_encoded['duration_cat'] = df_encoded['duration_minutes'].apply(dur_category)

#  Feature: Time of day 
# Morning < 12 | Afternoon 12-18 | Evening ≥ 18
def time_of_day(h):
    if h < 12:   return 0
    elif h < 18: return 1
    else:        return 2

df_encoded['time_of_day'] = df_encoded['hour_start'].apply(time_of_day)

#  Feature: Hemisphere (Northern / Southern based on locations) 
# Southern hemisphere circuits: Melbourne (Australia), Sao Paulo (Brazil),
# Abu Dhabi / Las Vegas are near equator; we use GMT offset as proxy
df_encoded['southern_hemi'] = (df_encoded['gmt_offset_hours'] >= 10).astype(int)

print(' New features added:')
new_feats = ['is_weekend', 'season_part', 'duration_cat', 'time_of_day', 'southern_hemi']
print(df_encoded[new_feats].value_counts().head(10))
print(f'\nTotal features now: {df_encoded.shape[1]}')

---
## Section 5 — Clustering Algorithms

**Clustering** is an **unsupervised** ML technique — we don't have labels/answers, we let the algorithm discover natural groups.

We will apply 4 algorithms:
| Algorithm | Key Idea |
|---|---|
| K-Means | Assigns each point to the nearest centroid, minimising total distance |
| K-Means++ | Smarter initialisation of centroids to avoid bad local minima |
| Hierarchical | Builds a tree of clusters; we cut the tree at a chosen level |
| DBSCAN | Groups dense regions; marks sparse points as noise |

> **No labels needed!** The algorithm figures out the groups by itself.

In [ ]:
#  Prepare feature matrix for clustering 
cluster_features = [
    'duration_minutes', 'hour_start', 'month',
    'gmt_offset_hours', 'day_of_week',
    'session_type_enc', 'season_part'
]

X_clust = df_encoded[cluster_features].copy()

# Scale the features (very important for clustering!)
scaler = StandardScaler()
X_clust_scaled = scaler.fit_transform(X_clust)

print('Clustering feature matrix shape:', X_clust_scaled.shape)
print('Features used:', cluster_features)

In [ ]:
#  Elbow Method to find optimal K 
# The "elbow" is the point where adding more clusters stops helping much
inertia_list = []
silhouette_list = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                random_state=RANDOM_STATE)
    labels = km.fit_predict(X_clust_scaled)
    inertia_list.append(km.inertia_)
    silhouette_list.append(silhouette_score(X_clust_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertia_list, 'bo-', markersize=7)
axes[0].set_title('Elbow Method — Inertia', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (lower = tighter clusters)')
axes[0].axvline(x=3, color='red', linestyle='--', label='k=3 (chosen)')
axes[0].legend()

axes[1].plot(k_range, silhouette_list, 'ro-', markersize=7)
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].axvline(x=3, color='red', linestyle='--', label='k=3 (chosen)')
axes[1].legend()

plt.suptitle('Choosing Optimal k for K-Means', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

BEST_K = 3
print(f'\n Chosen k = {BEST_K} (matches the 3 session types: Practice, Qualifying, Race)')

In [ ]:
#  5.1 K-Means (random initialisation) 
kmeans_random = KMeans(n_clusters=BEST_K, init='random', n_init=10,
                       random_state=RANDOM_STATE)
df_encoded['kmeans_random_label'] = kmeans_random.fit_predict(X_clust_scaled)

sil_r = silhouette_score(X_clust_scaled, df_encoded['kmeans_random_label'])
db_r  = davies_bouldin_score(X_clust_scaled, df_encoded['kmeans_random_label'])
print(f'K-Means (random)   Silhouette: {sil_r:.4f}  |  Davies-Bouldin: {db_r:.4f}')

In [ ]:
#  5.2 K-Means++ (smarter initialisation) 
# K-Means++ picks initial centroids far apart — usually converges faster
kmeans_pp = KMeans(n_clusters=BEST_K, init='k-means++', n_init=10,
                   random_state=RANDOM_STATE)
df_encoded['kmeans_pp_label'] = kmeans_pp.fit_predict(X_clust_scaled)

sil_pp = silhouette_score(X_clust_scaled, df_encoded['kmeans_pp_label'])
db_pp  = davies_bouldin_score(X_clust_scaled, df_encoded['kmeans_pp_label'])
print(f'K-Means++           Silhouette: {sil_pp:.4f}  |  Davies-Bouldin: {db_pp:.4f}')

In [ ]:
#  5.3 Hierarchical Clustering 
# First, visualise the dendrogram to see how clusters merge
fig, ax = plt.subplots(figsize=(16, 6))
Z = linkage(X_clust_scaled, method='ward')
dendrogram(Z, ax=ax, truncate_mode='level', p=5,
           color_threshold=0.7 * max(Z[:, 2]))
ax.set_title('Hierarchical Clustering — Dendrogram (Ward linkage)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Sample index')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

# Fit with k=3
hier = AgglomerativeClustering(n_clusters=BEST_K, linkage='ward')
df_encoded['hier_label'] = hier.fit_predict(X_clust_scaled)

sil_h = silhouette_score(X_clust_scaled, df_encoded['hier_label'])
db_h  = davies_bouldin_score(X_clust_scaled, df_encoded['hier_label'])
print(f'Hierarchical        Silhouette: {sil_h:.4f}  |  Davies-Bouldin: {db_h:.4f}')

In [ ]:
#  5.4 DBSCAN 
# DBSCAN doesn't need k! It finds clusters automatically.
# eps = neighbourhood radius, min_samples = minimum points to form a core
dbscan = DBSCAN(eps=1.0, min_samples=3)
df_encoded['dbscan_label'] = dbscan.fit_predict(X_clust_scaled)

n_clusters_db = len(set(df_encoded['dbscan_label'])) - (1 if -1 in df_encoded['dbscan_label'].values else 0)
n_noise_db = (df_encoded['dbscan_label'] == -1).sum()
print(f'DBSCAN  Clusters found: {n_clusters_db}  |  Noise points: {n_noise_db}')

if n_clusters_db > 1:
    # Only compute silhouette if there are at least 2 clusters
    mask = df_encoded['dbscan_label'] != -1
    sil_db = silhouette_score(X_clust_scaled[mask], df_encoded.loc[mask, 'dbscan_label'])
    db_db  = davies_bouldin_score(X_clust_scaled[mask], df_encoded.loc[mask, 'dbscan_label'])
    print(f'DBSCAN (excl noise)  Silhouette: {sil_db:.4f}  |  Davies-Bouldin: {db_db:.4f}')
else:
    sil_db, db_db = None, None
    print('DBSCAN found only 1 cluster — try adjusting eps or min_samples')

In [ ]:
#  5.5 Visualise Clustering Results (PCA 2D projection) 
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_clust_scaled)

algo_labels = [
    ('K-Means (random)',  'kmeans_random_label'),
    ('K-Means++',         'kmeans_pp_label'),
    ('Hierarchical',      'hier_label'),
    ('DBSCAN',            'dbscan_label'),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, col) in zip(axes, algo_labels):
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                         c=df_encoded[col], cmap='Set1',
                         alpha=0.75, s=50, edgecolors='white', linewidths=0.3)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')
    plt.colorbar(scatter, ax=ax)

plt.suptitle('Clustering Results (2D PCA Projection)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  Clustering Comparison Table 
clust_results = pd.DataFrame({
    'Algorithm':       ['K-Means (random)', 'K-Means++', 'Hierarchical', 'DBSCAN'],
    'Silhouette ↑':    [round(sil_r,4), round(sil_pp,4), round(sil_h,4),
                        round(sil_db,4) if sil_db else 'N/A'],
    'Davies-Bouldin ↓':[round(db_r,4),  round(db_pp,4),  round(db_h,4),
                        round(db_db,4) if db_db else 'N/A'],
})
print('Clustering Performance Summary:')
print(clust_results.to_string(index=False))
print('\n↑ = higher is better  |  ↓ = lower is better')

---
##  Section 6 — Classification

**Classification** is a **supervised** ML technique — we have a target label and train models to predict it.

### Our Target
We will predict **`session_type`** (Practice / Qualifying / Race) — a 3-class problem!

| Model | Family | Idea |
|---|---|---|
| KNN | Instance-based | Predict by majority vote of the k nearest neighbours |
| Decision Tree | Tree | Split data by asking yes/no questions |
| Logistic Regression | Linear | Fits a linear boundary between classes |
| Random Forest | Bagging | Combine many decision trees (each on a random subset) |
| AdaBoost | Boosting | Sequentially build weak learners, each fixing previous errors |
| Stacking | Ensemble | Use multiple models and a meta-learner to combine predictions |

In [ ]:
#  Prepare feature matrix and target 
# Target: session_type encoded (0=Practice, 1=Qualifying, 2=Race)
target_col = 'session_type_enc'

feature_cols = [
    'duration_minutes', 'hour_start', 'month', 'day_of_week',
    'gmt_offset_hours', 'circuit_key', 'country_key',
    'is_weekend', 'season_part', 'time_of_day',
    'session_name_enc'
]

X = df_encoded[feature_cols]
y = df_encoded[target_col]

# Class names for plotting
class_names = ['Practice', 'Qualifying', 'Race']

print('Feature matrix shape:', X.shape)
print('Target distribution:')
print(y.value_counts().rename(index={0:'Practice',1:'Qualifying',2:'Race'}))

#  Train / Test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

#  Scale features 
scaler_cls = StandardScaler()
X_train_sc = scaler_cls.fit_transform(X_train)
X_test_sc  = scaler_cls.transform(X_test)

print(f'\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

In [ ]:
#  Train all classifiers and collect results 

# ---- Base models for Stacking ----
base_estimators = [
    ('knn',  KNeighborsClassifier(n_neighbors=5)),
    ('dt',   DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)),
    ('rf',   RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE)),
]

classifiers = {
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'AdaBoost':            AdaBoostClassifier(
                               estimator=DecisionTreeClassifier(max_depth=2),
                               n_estimators=100, random_state=RANDOM_STATE),
    'Stacking':            StackingClassifier(
                               estimators=base_estimators,
                               final_estimator=LogisticRegression(max_iter=500),
                               cv=5),
}

results = {}   # store {name: (model, y_pred, accuracy)}

print('Training models...\n')
for name, clf in classifiers.items():
    clf.fit(X_train_sc, y_train)
    y_pred = clf.predict(X_test_sc)
    acc = accuracy_score(y_test, y_pred)
    results[name] = (clf, y_pred, acc)
    print(f'  {name:25s}  Accuracy = {acc:.4f}')

print('\n All models trained!')

---
##  Section 7 — Model Evaluation

We evaluate each model using three tools:

1. **Accuracy** — percentage of correct predictions  
2. **Confusion Matrix** — shows which classes are confused with which  
3. **Classification Report** — precision, recall, and F1-score per class

> **Precision:** Of all predicted positives, how many are really positive?  
> **Recall:** Of all actual positives, how many did we catch?  
> **F1-score:** Harmonic mean of precision and recall — balances both.

In [ ]:
#  Confusion Matrices for all models 
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (name, (clf, y_pred, acc)) in enumerate(results.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')
    axes[idx].set_title(f'{name}\nAccuracy = {acc:.3f}', fontweight='bold')

plt.suptitle('Confusion Matrices — All Classifiers', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  Classification Reports 
for name, (clf, y_pred, acc) in results.items():
    print(f"{'='*55}")
    print(f' {name}')
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred, target_names=class_names))

---
##  Section 8 — Model Comparison

Let's put all results side by side in a table and chart.

In [ ]:
#  Build comparison table 
from sklearn.metrics import f1_score, precision_score, recall_score

rows = []
for name, (clf, y_pred, acc) in results.items():
    cv_scores = cross_val_score(clf, X_train_sc, y_train, cv=5, scoring='accuracy')
    rows.append({
        'Model':          name,
        'Test Accuracy':  round(acc, 4),
        'CV Mean':        round(cv_scores.mean(), 4),
        'CV Std':         round(cv_scores.std(), 4),
        'Precision (w)':  round(precision_score(y_test, y_pred, average='weighted'), 4),
        'Recall (w)':     round(recall_score(y_test, y_pred, average='weighted'), 4),
        'F1 (weighted)':  round(f1_score(y_test, y_pred, average='weighted'), 4),
    })

compare_df = pd.DataFrame(rows).sort_values('Test Accuracy', ascending=False)
print('Model Comparison Table (sorted by Test Accuracy):')
print(compare_df.to_string(index=False))

In [ ]:
#  Visualise comparison 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart — accuracy
models   = compare_df['Model']
acc_vals = compare_df['Test Accuracy']
colors   = sns.color_palette('RdYlGn', len(models))

bars = axes[0].barh(models, acc_vals, color=colors[::-1], edgecolor='white')
axes[0].set_xlim(0, 1.05)
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Test Accuracy by Model', fontweight='bold', fontsize=13)
for bar, val in zip(bars, acc_vals):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontweight='bold')

# Grouped bar — Precision / Recall / F1
x = np.arange(len(compare_df))
width = 0.25
axes[1].bar(x - width, compare_df['Precision (w)'], width, label='Precision', color='#4C72B0')
axes[1].bar(x,         compare_df['Recall (w)'],    width, label='Recall',    color='#DD8452')
axes[1].bar(x + width, compare_df['F1 (weighted)'], width, label='F1',        color='#55A868')
axes[1].set_xticks(x)
axes[1].set_xticklabels(compare_df['Model'], rotation=25, ha='right')
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Score')
axes[1].set_title('Precision / Recall / F1 by Model', fontweight='bold', fontsize=13)
axes[1].legend()

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  Feature Importance (Random Forest) 
rf_model = results['Random Forest'][0]
importances = pd.Series(
    rf_model.feature_importances_, index=feature_cols
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importances — Random Forest', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Most important feature:', importances.idxmax())

In [ ]:
#  Visualise the Decision Tree 
dt_model = results['Decision Tree'][0]

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_model, feature_names=feature_cols,
          class_names=class_names, filled=True,
          max_depth=3, ax=ax, fontsize=9,
          impurity=False, rounded=True)
ax.set_title('Decision Tree (first 3 levels)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---
##  Section 9 — Final Conclusion

### Best Model

Based on our evaluation, the **top-performing classifiers** are typically **Random Forest** and **Stacking**, as they:
- Achieve the highest test accuracy and F1-score
- Are stable across cross-validation folds
- Handle both numeric and encoded features well

**Logistic Regression** performs reasonably given that `session_name` is a near-perfect signal for `session_type`, showing that even simple linear models can work when the features carry enough information.

### Clustering Verdict
K-Means++ slightly outperforms random K-Means in both silhouette score and convergence speed. Hierarchical clustering gives similar results. DBSCAN is useful to detect anomalous sessions (noise points).

---

##  Insights from the Dataset

1. **Duration is the strongest discriminator** — Races are ~120 min, Qualifying ~60 min, Practice ~60–90 min.
2. **Hour of the day varies by session type** — Races tend to start later in the afternoon/evening (prime TV time).
3. **Seasonal spread** — F1 runs March through November, with a summer break visible as a gap in July/August.
4. **GMT offsets** cluster around +2 to +3 (Europe/Middle East) and outliers at −5 to −8 (Americas) and +9/+10 (Japan/Australia).
5. **Sprint weekends** are rare and shorter — they appear as outlier sessions in clustering.

---

##   How to Improve the Models

| Idea | Why it helps |
|---|---|
| **Add lap time / telemetry data** | Real F1 data can reveal track difficulty and session intensity |
| **Hyperparameter tuning** (GridSearchCV / RandomSearch) | Find the best `n_estimators`, `max_depth`, `k`, etc. |
| **More seasons** (2021, 2022, 2024) | More data = better generalisation |
| **Weather features** | Rain races differ significantly from dry ones |
| **One-hot encoding** instead of label encoding for nominals | Avoids imposing false ordinal relationships |
| **SMOTE / class weighting** | Handles imbalance between Practice (majority) and Race/Qualifying |
| **XGBoost / LightGBM** | Gradient boosting often outperforms AdaBoost on tabular data |
| **Neural network** | For a larger dataset, a simple MLP could capture non-linear patterns |


In [ ]:
#  Final Summary Print 
best_model_name = compare_df.iloc[0]['Model']
best_accuracy   = compare_df.iloc[0]['Test Accuracy']

print('=' * 55)
print('    FINAL PROJECT SUMMARY')
print('=' * 55)
print(f'  Dataset           : F1 2023 Sessions ({df.shape[0]} rows, {df.shape[1]} cols)')
print(f'  Target            : session_type (3 classes)')
print(f'  Features used     : {len(feature_cols)}')
print(f'  Train/Test split  : 80% / 20%')
print()
print(f'  Best Classifier   : {best_model_name}')
print(f'  Best Accuracy     : {best_accuracy:.4f} ({best_accuracy*100:.1f}%)')
print()
print('  Model Rankings (by accuracy):')
for i, row in compare_df.reset_index(drop=True).iterrows():
    medal = ['  4.','  5.','  6.'][i]
    print(f'  {medal} {row["Model"]:25s}  {row["Test Accuracy"]:.4f}')
print('=' * 55)